# Estimate cardinality and selectivity

**Goal:** compare exact counts, fresh samples and a reusable sample.

Start Docker and run all cells in order with the BONDI Python environment.
The first code cell creates the named `notebook-estimation` deployment and loads the
eight fictional businesses into `demo.business`.
No external data or connection configuration is needed. If a port is occupied,
change the port arguments in the setup cell before the first run.

The final cell removes the deployment, including its data volume and saved configuration.
Rerunning setup creates a fresh deployment and reloads the demo documents. If you
interrupt execution, run the final cleanup cell manually. Query examples read the demo data; the setup cell writes it.

Companion guide: [cardinality estimation](../docs/mongodb-estimation.md).

In [ ]:
from bondi.deployment import deploy_mongodb

deployment = deploy_mongodb(name="notebook-estimation", port=27122)
deployment.load_demo_data()


In [ ]:
from bondi.mongodb import MongoCollectionRef, MongoExecutor, MongoFindQuery, MongoSampleEstimator

target = MongoCollectionRef("demo", "business")
query = MongoFindQuery(target, {"stars": {"$gte": 4}}, limit=1)

## Exact evaluation versus a fresh sample

Cardinality counts filter matches. Selectivity divides that count by the population.
Both ignore the query's `limit=1`. Omitting sample controls evaluates the full population;
`sample_ratio=0.5` draws a fresh sample for each estimate.

In [ ]:
with MongoExecutor.from_uri(deployment.uri, timeout_seconds=10) as executor:
    with MongoSampleEstimator(executor, target) as estimator:
        exact = estimator.estimate(query)
    with MongoSampleEstimator(executor, target, sample_ratio=0.5) as estimator:
        fresh = estimator.estimate(query)

for name, estimate in [("Full population", exact), ("Fresh sample", fresh)]:
    print(name)
    print("  Cardinality:", estimate.cardinality, "Selectivity:", estimate.selectivity)
    print("  Sample matches:", estimate.sample_matches, "Sample size:", estimate.sample_size)
assert exact.cardinality > query.limit

## Reuse one sample

Use `sample_size` or `sample_ratio`, never both. `persist=True` creates an owned
sample collection so repeated checks use the same data. Nested context managers
close the sample before its executor, including if estimation fails.

In [ ]:
with MongoExecutor.from_uri(deployment.uri, timeout_seconds=10) as executor:
    with MongoSampleEstimator(executor, target, sample_size=4, persist=True) as estimator:
        print("Temporary collection:", estimator.sample_target)
        first = estimator.estimate(query)
        second = estimator.estimate(query)
        print(first)
        assert first == second
    print("Sample removed:", estimator.sample_target is None)
    assert estimator.sample_target is None

## Try next

Change the stars threshold and compare sample estimates with the exact count.
Repeat the fresh-sample cell: the result may change. MongoDB sampling is not
controlled by a Python random seed.

Keep source data stable during an experiment. Zero sample matches do not prove
zero population matches. Next, use an estimator to constrain generated queries in
[the workload generation notebook](workload_generation.ipynb).
Manual preparation and cleanup are covered in the
[API reference](../docs/reference/mongodb/estimation.md).

## Remove the deployment

Remove the database container, data volume and saved configuration. All executors
and temporary samples have already been closed by their context managers.
Run the setup cell again before continuing with database examples.


In [ ]:
deployment.remove(delete_data=True)
print("Deployment and data removed.")